# Лабораторная работа №5

## Тема
Сравнение ансамблевых методов классификации.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.metrics import f1_score

RANDOM_STATE = 42
DATA_PATH = "../data/breast_cancer_wisconsin/data.csv"

df = pd.read_csv(DATA_PATH)
df = df.drop(columns=["id", "Unnamed: 32"], errors="ignore")
df["diagnosis"] = df["diagnosis"].replace({"M": 1, "B": 0})
df = df[df["diagnosis"].isin([0, 1])].copy()
df["diagnosis"] = df["diagnosis"].astype(int)

X = df.drop(columns=["diagnosis"])
y = df["diagnosis"]

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# Для бустинговых моделей удобнее работать с числовой матрицей
X_train_p = preprocessor.fit_transform(X_train, y_train)
X_test_p = preprocessor.transform(X_test)

In [2]:
models = {
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE),
    "ExtraTrees": ExtraTreesClassifier(n_estimators=300, random_state=RANDOM_STATE),
    "AdaBoost": AdaBoostClassifier(n_estimators=200, random_state=RANDOM_STATE),
    "GradientBoosting": GradientBoostingClassifier(random_state=RANDOM_STATE)
}

results = []
for name, model in models.items():
    model.fit(X_train_p, y_train)
    pred = model.predict(X_test_p)
    results.append({
        "model": name,
        "f1": f1_score(y_test, pred)
    })

pd.DataFrame(results).sort_values(by="f1", ascending=False)

,model,f1
1,ExtraTrees,0.975610
0,RandomForest,0.962963
2,AdaBoost,0.962963
3,GradientBoosting,0.950000


## Выводы

Построены четыре ансамблевые модели, включая две из группы бэггинга (`RandomForest`, `ExtraTrees`) и две бустинговые (`AdaBoost`, `GradientBoosting`).
Сравнение выполнено по метрике `f1`, после чего определена наиболее эффективная модель на тестовой выборке.